In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 如果缺少库，运行以下命令安装
# !pip install requests pandas numpy folium matplotlib

print("✅ 环境配置完成")

In [ ]:
API_KEY = "你的真实Key"  
# 验证 Key 是否有效（天气 API 测试）
test_url = f"https://restapi.amap.com/v3/weather/weatherInfo?key={API_KEY}&city=310000"  # 上海城市代码 310000
try:
    response = requests.get(test_url, timeout=5).json()
    if response["status"] == "1":
        print("✅ API Key 有效，可以开始数据采集")
    else:
        print(f"❌ API Key 无效或已过期，错误信息：{response.get('info', '未知错误')}")
except Exception as e:
    print(f"❌ 网络请求失败，请检查网络连接：{e}")

In [ ]:
def get_pois_by_location(location, keywords, radius=1000, page=1):
    """
    使用高德地图周边搜索API获取POI数据。
    返回:
        - 成功：POI列表
        - 无数据：[]
        - 致命错误（额度耗尽/Key无效）："FATAL"
    """
    url = "https://restapi.amap.com/v3/place/around"
    params = {
        "key": API_KEY,
        "location": location,
        "keywords": keywords,
        "radius": radius,
        "offset": 25,          # 每页最多25条
        "page": page,
        "output": "json"
    }
    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        if data["status"] == "1" and data["pois"]:
            return data["pois"]
        else:
            error_code = data.get("infocode", "")
            error_info = data.get("info", "")
            # 致命错误码：10001(Key无效), 10002(Key不存在), 10003(配额超限), 10004(日调用超限)
            if error_code in ["10001", "10002", "10003", "10004"]:
                print(f"❌ 致命错误: {error_info}，程序终止。")
                return "FATAL"
            return []
    except Exception as e:
        print(f"请求异常: {e}")
        return []

In [ ]:
# 上海外滩中心坐标（南京东路与外滩交汇处附近）
base_lng, base_lat = 121.4877, 31.2358
grid_step = 0.015          # 网格步长，约1.5公里
search_radius = 1500       # 搜索半径，单位米

all_pois = []
fatal_flag = False

print(f"开始以外滩为中心进行网格搜索，请稍候...")
for i in range(-2, 3):       # 经度方向，覆盖约6公里范围
    if fatal_flag:
        break
    for j in range(-2, 3):   # 纬度方向
        if fatal_flag:
            break
        location = f"{base_lng + i * grid_step},{base_lat + j * grid_step}"
        print(f"正在搜索区域: {location}")
        page = 1
        while page <= 100:
            pois = get_pois_by_location(location, "咖啡店", radius=search_radius, page=page)
            if pois == "FATAL":
                fatal_flag = True
                break
            if not pois:
                break
            all_pois.extend(pois)
            page += 1
            time.sleep(0.1)      # 分页间延迟，避免QPS超限
        time.sleep(0.3)          # 网格间延迟，更安全

if fatal_flag:
    print("⚠️ 由于额度耗尽或 Key 无效，数据采集提前终止。")
else:
    print(f"✅ 数据采集完成，共获取 {len(all_pois)} 条 POI 数据。")

In [ ]:
if len(all_pois) > 0:
    # 提取关键字段
    df = pd.DataFrame(all_pois)
    # 确保必要的列存在
    if 'shopinfo' not in df.columns:
        df['shopinfo'] = ''
    df = df[['name', 'address', 'location', 'adname', 'type', 'shopinfo']]
    
    # 拆分经纬度
    df['lng'] = df['location'].str.split(',').str[0].astype(float)
    df['lat'] = df['location'].str.split(',').str[1].astype(float)
    
    # 标记是否为连锁品牌（简单规则，可根据需要增删）
    brands = ['星巴克', '瑞幸', 'Costa', '太平洋咖啡', 'Tims', 'Manner', 'M Stand', 'Peets', 'LAVAZZA', '代数学家']
    df['is_chain'] = df['name'].apply(lambda x: any(brand in x for brand in brands))
    
    # 去重（同一经纬度只保留一条）
    df = df.drop_duplicates(subset=['lng', 'lat']).reset_index(drop=True)
    
    print(f"✅ 数据清洗完成，有效数据 {len(df)} 条。")
    print(f"   其中连锁品牌 {df['is_chain'].sum()} 家，独立店铺 {len(df) - df['is_chain'].sum()} 家。")
    print(df.head())
else:
    print("⚠️ 未获取到任何数据，请检查 API Key 或网络。")
    df = pd.DataFrame()

In [ ]:
if len(df) > 0:
    # 以外滩为中心创建地图
    map_center = [31.2358, 121.4877]  # [纬度, 经度]
    m = folium.Map(location=map_center, zoom_start=14, tiles='OpenStreetMap')
    
    # 添加热力图图层
    heat_data = [[row['lat'], row['lng']] for _, row in df.iterrows()]
    HeatMap(heat_data, radius=15, blur=10).add_to(m)
    
    # 保存为 HTML 文件
    m.save("shanghai_bund_coffee_heatmap.html")
    print("✅ 热力图已保存为 shanghai_bund_coffee_heatmap.html，双击即可在浏览器中打开查看。")
else:
    print("⚠️ 无数据，无法生成热力图。")

In [ ]:
if len(df) > 0 and df['is_chain'].sum() > 0:
    # 统计连锁品牌数量
    chain_df = df[df['is_chain']]['name'].value_counts().head(10)
    
    plt.figure(figsize=(10, 6))
    chain_df.plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title('上海外滩咖啡店连锁品牌 Top10 分布', fontsize=14)
    plt.xlabel('品牌名称')
    plt.ylabel('门店数量')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("shanghai_bund_brand_distribution.png", dpi=150)
    plt.show()
    print("✅ 品牌分布图已保存为 shanghai_bund_brand_distribution.png")
else:
    print("⚠️ 连锁品牌数据不足，跳过可视化。")

In [ ]:
if len(df) > 0:
    district_counts = df['adname'].value_counts().head(10)
    
    plt.figure(figsize=(10, 6))
    district_counts.plot(kind='bar', color='darkorange', edgecolor='black')
    plt.title('上海外滩咖啡店行政区 Top10 分布', fontsize=14)
    plt.xlabel('行政区')
    plt.ylabel('门店数量')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("shanghai_bund_district_distribution.png", dpi=150)
    plt.show()
    print("✅ 行政区分布图已保存为 shanghai_bund_district_distribution.png")

In [ ]:
if len(df) > 0:
    # 导出 CSV
    df.to_csv("shanghai_bund_coffee_shops.csv", index=False, encoding='utf-8-sig')
    print("✅ 原始数据已导出为 shanghai_bund_coffee_shops.csv")
    
    # 输出统计摘要
    print("\n" + "="*40)
    print("📊 上海外滩咖啡店数据分析摘要")
    print("="*40)
    print(f"总门店数：{len(df)} 家")
    print(f"连锁品牌占比：{df['is_chain'].mean():.1%}")
    print(f"覆盖行政区数：{df['adname'].nunique()} 个")
    if len(df['adname'].value_counts()) > 0:
        print(f"门店最多的行政区：{df['adname'].value_counts().index[0]}（{df['adname'].value_counts().iloc[0]} 家）")
    if df['is_chain'].sum() > 0:
        print(f"门店最多的连锁品牌：{df[df['is_chain']]['name'].value_counts().index[0]}（{df[df['is_chain']]['name'].value_counts().iloc[0]} 家）")